In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests

In [2]:
data_path = Path(os.environ["DATA_PATH"])

In [3]:
unwanted_names = [
    "SIN DATOS",
]
df = (
    pd.read_csv(data_path / "conocimientoCliente-email20260330_empleos2.csv")
    .drop(columns=["Unnamed: 0"])
    .loc[lambda df: ~df["Empresa"].isin(unwanted_names)]
)

In [ ]:
xmin, ymin, xmax, ymax = (
    gpd.read_file(data_path / "initial" / "lim_aumxl24")
    .to_crs("EPSG:4326")
    .total_bounds.round(3)
    .tolist()
)


In [58]:
ENDPOINT_URL = f"https://atlas.microsoft.com/search/fuzzy/batch/sync/json?api-version=1.0&subscription-key={os.environ['AZURE_MAPS_PK']}&countrySet=MX&language=es&topLeft={ymax},{xmin}&btmRight={ymin},{xmax}"

In [65]:
names = df["Empresa"].unique()
batch_items = {"batchItems": [{"query": f"?query={name}"} for name in names]}
test = {
    "batchItems": batch_items["batchItems"][:10],
}

response = requests.post(
    ENDPOINT_URL,
    json=test,
    headers={
        "Content-Type": "application/json",
        "Accept-Language": "es",
        "subscription-key": os.environ["AZURE_MAPS_PK"],
    },
    timeout=60,
)

In [66]:
response_dict = response.json()

In [67]:
test["batchItems"][4]

{'query': '?query=RESTAURANTE MINUZ'}

In [76]:
response_dict["batchItems"][0]["response"]["results"]

[{'type': 'Street',
  'id': '8NftL_HcscfROrEAhKlg2g',
  'score': 4.5216622353,
  'address': {'streetName': 'Jirón Cecilio Hernández',
   'municipality': 'Callería',
   'countrySecondarySubdivision': 'Coronel Portillo',
   'countrySubdivision': 'Ucayali',
   'countrySubdivisionName': 'Ucayali',
   'countrySubdivisionCode': 'UCA',
   'postalCode': '25001',
   'countryCode': 'PE',
   'country': 'Perú',
   'countryCodeISO3': 'PER',
   'freeformAddress': 'Jirón Cecilio Hernández Callería, 25001',
   'localName': 'Callería'},
  'position': {'lat': -8.3904, 'lon': -74.530728},
  'viewport': {'topLeftPoint': {'lat': -8.39006, 'lon': -74.53127},
   'btmRightPoint': {'lat': -8.39069, 'lon': -74.53027}}},
 {'type': 'POI',
  'id': '-Q_9Gl42AXAujzAuhW1CJg',
  'score': 4.3663377762,
  'info': 'search:ta:840128000573211-US',
  'poi': {'name': 'Cecilio Hernandez MD',
   'phone': '+1 813-870-3979',
   'categorySet': [{'id': 9373}],
   'categories': ['doctor'],
   'classifications': [{'code': 'DOCTOR',


In [25]:
names

array(['MISSAEL CECILIO HERNANDEZ', 'FOOTPRINT MX S DE RL DE CV',
       'MEDTRONIC', ..., 'JOSE LUIS LOPEZ NUÑEZ',
       'FERRETERIA NUEVO REFUGIO', 'PAN WEBERS SA DE CV'],
      shape=(1537,), dtype=object)